# Proyecto 1
## Reporte de resultados sobre la base de datos Work at olist


### Integrantes
- Laura Berenice Badillo Santos: berenicebadillo10@ciencias.unam.mx
- Eduardo Sánchez Benítez: eduardo_sb@ciencias.unam.mx
- Juan Alberto Jaimes Romero: yoselerratil@gmail.com
- Juan Manuel Sóstenes Sánchez: juansossan@ciencias.unam.mx
- Angel Noriega Cabrera: angelnoriega@ciencias.unam.mx


## Comprender los datos

Antes que empezemos a transformar la información y manipularla tenemos que explorar las bases y revisar que información nos puede servir para responder

In [1]:
### Reconocemos donde esta la carpeta que contiene la data que vamos a estar utilizando
from pathlib import Path
import os
import sys
from IPython.display import display
import numpy as np
import pandas as pd

# Estructura real del proyecto:
# DMEMCD_2026/Python/*.ipynb y Python/olist_tools.py
# DMEMCD_2026/work-at-olist-data/datasets/*.csv
# Funciona si el kernel arranca en la raíz o en alguna subcarpeta.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Python" / "olist_tools.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No encuentro Python/olist_tools.py. "
        "Abre en VS Code la carpeta DMEMCD_2026 y selecciona su kernel."
    )
PYTHON_DIR = ROOT / "Python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
from olist_tools import (read_tables, build_order_level, build_item_level,
                        table_inventory, missing_profile, safe_minmax, safe_zscore)

def localizar_directorio(requeridos):
    candidatos = []
    definida = os.environ.get("OLIST_DATA_DIR")
    if definida:
        candidatos.append(Path(definida).expanduser())
    candidatos.append(Path("~/projects/work-at-olist-data/datasets").expanduser())
    ubicacion = Path.cwd().resolve()
    for base in (ubicacion, *ubicacion.parents):
        candidatos.extend([
            base / "work-at-olist-data" / "datasets",
            base / "datasets",
            base.parent / "work-at-olist-data" / "datasets",
        ])
    vistos = set()
    for ruta in candidatos:
        clave = str(ruta)
        if clave in vistos:
            continue
        vistos.add(clave)
        if all((ruta / nombre).is_file() for nombre in requeridos):
            return ruta
    raise FileNotFoundError(
        "No encontré los CSV requeridos. Define OLIST_DATA_DIR o clona "
        "https://github.com/olist/work-at-olist-data junto al proyecto."
    )

REQUERIDOS = (
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
)
DATA_DIR = localizar_directorio(REQUERIDOS)
# read_tables añade work-at-olist-data/datasets a la raíz que recibe.
TABLAS, ORIGEN = read_tables(DATA_DIR.parent.parent)
print("ORIGEN:", ORIGEN)
print("Directorio raíz:", ROOT)
print("Directorio de datos:", DATA_DIR)


ORIGEN: CSV reales suministrados: work-at-olist-data/datasets
Directorio raíz: /home/alberto/projects/DMEMCD_2026_equipo_15
Directorio de datos: /home/alberto/projects/work-at-olist-data/datasets


In [2]:
### Cargamos las bases de datos para poder comprenderlas y saber cual usar para poder dar una análisis
### integro


orders = pd.read_csv(
    DATA_DIR /
    "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
)

print("Dimensiones:", orders.shape)
display(orders.head())

Dimensiones: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
### Generaremos un análisis de las siguientes tablas:

### Generaremos un análisis de las siguientes tablas para comprender su estructura y contenido.

customers = pd.read_csv(
    DATA_DIR /
    "olist_customers_dataset.csv"
)

geo = pd.read_csv(
    DATA_DIR /
    "olist_geolocation_dataset.csv"
)

products = pd.read_csv(
    DATA_DIR /
    "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    DATA_DIR /
    "olist_sellers_dataset.csv"
)

In [4]:
print("Dimensiones:", customers.shape)
display(customers.head())

customers.info()

print("Número de customer_id:", customers["customer_id"].nunique())
print("Número de clientes únicos:", customers["customer_unique_id"].nunique())

Dimensiones: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
Número de customer_id: 99441
Número de clientes únicos: 96096


### Comprensión de la tabla `customers`

La tabla `customers` cuenta con **99,441 registros y 5 variables**, las cuales contienen información de identificación y ubicación geográfica de los clientes.

Las variables disponibles son `customer_id`, `customer_unique_id`, `customer_zip_code_prefix`, `customer_city` y `customer_state`. En una primera inspección se observó que ninguna de estas variables presenta valores nulos.

Se identificaron **99,441 valores únicos de `customer_id`**, mientras que `customer_unique_id` presenta **96,096 clientes únicos**. Esta diferencia indica que un mismo cliente puede encontrarse asociado a más de un registro. Por esta razón, `customer_unique_id` será la variable más adecuada cuando se requiera calcular el número real de clientes.

La información geográfica de la tabla permitirá además analizar la distribución de los clientes por ciudad, estado y código postal y relacionar estos resultados con otras tablas del conjunto de datos.

In [5]:
print("Dimensiones:", geo.shape)
display(geo.head())

geo.info()

print("Número de códigos postales únicos:", geo["geolocation_zip_code_prefix"].nunique())
print("Número de ciudades únicas:", geo["geolocation_city"].nunique())
print("Número de estados únicos:", geo["geolocation_state"].nunique())

Dimensiones: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB
Número de códigos postales únicos: 19015
Número de ciudades únicas: 8011
Número de estados únicos: 27


### Comprensión de la tabla `geolocation`

La tabla `geolocation` contiene **1,000,163 registros y 5 variables**. Su principal función es proporcionar información geográfica asociada a los códigos postales, incluyendo coordenadas, ciudad y estado.

Las variables que contiene son:

- `geolocation_zip_code_prefix`: prefijo del código postal.
- `geolocation_lat`: latitud correspondiente a la ubicación geográfica.
- `geolocation_lng`: longitud correspondiente a la ubicación geográfica.
- `geolocation_city`: ciudad asociada a la ubicación.
- `geolocation_state`: estado asociado a la ubicación.

En cuanto a los tipos de datos, `geolocation_zip_code_prefix` está almacenada como tipo `int64`, las coordenadas `geolocation_lat` y `geolocation_lng` como `float64`, mientras que `geolocation_city` y `geolocation_state` son de tipo `object`.

Durante la inspección inicial **no se identificaron valores nulos** en ninguna de las cinco variables.

A pesar de contar con **1,000,163 registros**, únicamente existen **19,015 prefijos de códigos postales únicos**, **8,011 ciudades únicas** y **27 estados únicos**. Esto indica que un mismo código postal puede aparecer en múltiples registros, por lo que cada fila no debe interpretarse como una ubicación completamente independiente.

Esta tabla puede ser útil para complementar otras bases mediante el código postal y realizar análisis relacionados con la **distribución geográfica de clientes y vendedores**, así como representar sus ubicaciones mediante las coordenadas de latitud y longitud.

Será importante considerar la repetición de los códigos postales antes de relacionar esta tabla con otras bases, ya que una unión directa podría incrementar la cantidad de registros si un mismo código postal aparece varias veces.

In [6]:
print("Dimensiones:", products.shape)
display(products.head())

products.info()

print("Número de product_id:", products["product_id"].nunique())
print("Número de categorías únicas:", products["product_category_name"].nunique())

Dimensiones: (32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB
Número de product_id: 32951
Número de categorías únicas: 73


### Comprensión de la tabla `products`

La tabla `products` contiene **32,951 registros y 9 variables**. Su función es proporcionar información descriptiva y física sobre los productos disponibles en el conjunto de datos.

Las variables que contiene son:

- `product_id`: identificador único del producto.
- `product_category_name`: categoría a la que pertenece el producto.
- `product_name_lenght`: longitud del nombre del producto.
- `product_description_lenght`: longitud de la descripción del producto.
- `product_photos_qty`: cantidad de fotografías asociadas al producto.
- `product_weight_g`: peso del producto en gramos.
- `product_length_cm`: longitud del producto en centímetros.
- `product_height_cm`: altura del producto en centímetros.
- `product_width_cm`: ancho del producto en centímetros.

En cuanto a los tipos de datos, `product_id` y `product_category_name` se encuentran almacenadas como tipo `object`, mientras que las siete variables restantes son de tipo `float64`.

Se identificaron **32,951 valores únicos de `product_id`**, la misma cantidad que el número total de registros. Esto indica que cada fila corresponde a un producto diferente y que `product_id` puede funcionar como identificador único de la tabla.

También se identificaron **73 categorías de producto diferentes**.

A diferencia de las tablas analizadas anteriormente, esta tabla **sí presenta valores faltantes**. La variable `product_category_name`, junto con las variables relacionadas con la longitud del nombre, descripción y cantidad de fotografías, presenta **610 registros faltantes**. Las variables relacionadas con peso y dimensiones físicas presentan **2 valores faltantes** cada una.

Estos valores faltantes deberán analizarse posteriormente durante la etapa de limpieza para determinar si deben mantenerse, eliminarse o tratarse mediante alguna transformación, dependiendo del análisis que se realice.

Esta tabla puede ser útil para realizar análisis relacionados con **categorías de productos, características físicas, peso, dimensiones y cantidad de fotografías**, además de poder relacionarse posteriormente con otras tablas mediante `product_id`.

In [7]:
print("Dimensiones:", sellers.shape)
display(sellers.head())

sellers.info()

print("Número de seller_id:", sellers["seller_id"].nunique())
print("Número de códigos postales únicos:", sellers["seller_zip_code_prefix"].nunique())
print("Número de ciudades únicas:", sellers["seller_city"].nunique())
print("Número de estados únicos:", sellers["seller_state"].nunique())

Dimensiones: (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB
Número de seller_id: 3095
Número de códigos postales únicos: 2246
Número de ciudades únicas: 611
Número de estados únicos: 23


### Comprensión de la tabla `sellers`

La tabla `sellers` contiene **3,095 registros y 4 variables**. Su función es proporcionar información de identificación y ubicación geográfica de los vendedores presentes en el conjunto de datos.

Las variables que contiene son:

- `seller_id`: identificador único del vendedor.
- `seller_zip_code_prefix`: prefijo del código postal asociado al vendedor.
- `seller_city`: ciudad en la que se encuentra el vendedor.
- `seller_state`: estado en el que se encuentra el vendedor.

En cuanto a los tipos de datos, `seller_id`, `seller_city` y `seller_state` se encuentran almacenadas como tipo `object`, mientras que `seller_zip_code_prefix` se encuentra como tipo `int64`.

Durante la inspección inicial **no se identificaron valores nulos** en ninguna de las cuatro variables.

Se identificaron **3,095 valores únicos de `seller_id`**, exactamente la misma cantidad que el número total de registros. Esto indica que cada fila corresponde a un vendedor diferente y que `seller_id` puede utilizarse como identificador único de esta tabla.

En cuanto a la distribución geográfica, se encontraron **2,246 códigos postales únicos, 611 ciudades y 23 estados diferentes**. La diferencia entre el número de vendedores y códigos postales indica que distintos vendedores pueden encontrarse asociados al mismo código postal.

Esta tabla será útil para estudiar la **distribución geográfica de los vendedores** y, posteriormente, relacionar su ubicación con la de los clientes mediante variables como ciudad, estado o código postal. También podrá relacionarse con otras tablas del conjunto de datos mediante `seller_id`.